In [ ]:
# Import necessary libraries
import pandas as pd
from google.colab import files

# Upload the file in Colab
uploaded = files.upload()

# Load the dataset (uses the uploaded file name)
df = pd.read_excel(list(uploaded.keys())[0])

# Display basic structure
print("Dataset dimensions:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())
print("\nLast 5 rows:\n", df.tail())


Saving Log Transformed Data.xlsx to Log Transformed Data.xlsx
Dataset dimensions: (1321, 8)

Column names: ['Dates', 'US CDS', 'UK CDS', 'German CDS', 'Silver', 'Copper', 'Bitcoin', 'Gold']

Data types:
 Dates         datetime64[ns]
US CDS               float64
UK CDS               float64
German CDS           float64
Silver               float64
Copper               float64
Bitcoin              float64
Gold                 float64
dtype: object

First 5 rows:
        Dates   US CDS   UK CDS  German CDS    Silver    Copper    Bitcoin  \
0 2021-01-01  3.55897  3.55897    2.374999  3.276579  8.955319  10.283655   
1 2021-01-04  3.55897  3.55897    2.382782  3.305787  8.967504  10.342948   
2 2021-01-05  3.55897  3.55897    2.371271  3.314550  8.986509  10.427986   
3 2021-01-06  3.55897  3.55897    2.384165  3.315095  8.991002  10.489586   
4 2021-01-07  3.55897  3.55897    2.408835  3.300640  9.007857  10.589937   

       Gold  
0  7.548909  
1  7.571937  
2  7.575590  
3  7.559356  
4

In [ ]:
# Install if needed (run once in Colab)
!pip install statsmodels arch --quiet

# Imports
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import VECM, select_order
import warnings
warnings.filterwarnings('ignore')

# Assume df from Agent 2; set index
# Check if 'Dates' is a column before attempting to set it as index
if 'Dates' in df.columns:
    df['Dates'] = pd.to_datetime(df['Dates'])
    df.set_index('Dates', inplace=True)
elif df.index.name == 'Dates':
    # If 'Dates' is already the index, ensure it's a DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

# Select variables (log-transformed prices)
vars_list = ['Bitcoin', 'Gold', 'Silver', 'Copper', 'US CDS']
data = df[vars_list].dropna()

print("Data summary:")
print(data.describe())

# Engle-Granger cointegration tests (Bitcoin as Y, others as X)
pairs = [('Bitcoin', 'Gold'), ('Bitcoin', 'Silver'), ('Bitcoin', 'Copper'), ('Bitcoin', 'US CDS')]
coint_results = {}
for y, x in pairs:
    score, pvalue, _ = coint(data[y], data[x])
    coint_results[f'{y}~{x}'] = {'score': score, 'pvalue': pvalue}
    print(f"\nEngle-Granger {y} vs {x}: score={score:.4f}, pvalue={pvalue:.4f}")

# Lag selection for VECM (maxlags=5 for BIC)
lag_order = select_order(data, maxlags=5, deterministic="ci")
print("\nLag order selection (BIC):", lag_order.selected_orders)

# Fit VECM with lag=2 (as specified; k_ar_diff=2), rank=1 (assume 1 cointegrating relation)
vecm_model = VECM(data, k_ar_diff=2, coint_rank=1)
vecm_fit = vecm_model.fit()
print("\nVECM Summary:")
print(vecm_fit.summary())

# Full model outputs displayed above


Data summary:
           Bitcoin         Gold       Silver       Copper       US CDS
count  1321.000000  1321.000000  1321.000000  1321.000000  1321.000000
mean     10.763401     7.704047     3.304081     9.121178     2.643433
std       0.557920     0.267861     0.274410     0.101668     0.420144
min       9.657072     7.391637     2.877512     8.876265     2.255389
25%      10.290767     7.503620     3.134842     9.039849     2.324444
50%      10.756454     7.579960     3.220874     9.125781     2.324444
75%      11.162272     7.874252     3.421980     9.183996     2.995182
max      11.738153     8.511288     4.561218     9.493548     3.558970

Engle-Granger Bitcoin vs Gold: score=-1.5877, pvalue=0.7260

Engle-Granger Bitcoin vs Silver: score=-0.9754, pvalue=0.9074

Engle-Granger Bitcoin vs Copper: score=-2.0507, pvalue=0.5021

Engle-Granger Bitcoin vs US CDS: score=-2.4539, pvalue=0.2998

Lag order selection (BIC): {'aic': np.int64(1), 'bic': np.int64(1), 'hqic': np.int64(1), 'fpe': 